# Notebook di computer vision: un test set

Cinque modelli di visione, ognuno con la sua misura · Giorno 3 · Mattina

**Al termine sarete in grado di** contare TP, FP e FN di un modello di rilevamento e leggerne precisione e recall a
tre soglie, confrontare la recall di due gruppi guardando quanti veicoli ha ciascuno, misurare il tasso di errore di
un OCR e dire se un IBAN letto è esatto, misurare la recall@5 di una ricerca per immagini, e controllare che cosa
risponde un VLM quando la targa è troppo piccola.

## Il tema

Cinque modelli di visione su un piccolo test set: per ognuno un'esecuzione e una misura. I fotogrammi sono foto vere
di strade e parcheggi, con persone e targhe sfocate; i documenti del fascicolo e le targhe di fantasia sono sintetici.

| Parte | Che cosa si misura |
|---|---|
| 1. Rilevamento | Precisione e recall a tre soglie |
| 2. Gruppi | La recall di giorno e di notte, con il numero di veicoli |
| 3. OCR | Il tasso di errore sui caratteri, l'IBAN esatto o no, i caratteri incerti |
| 4. Ricerca per immagini | Una frase cerca tra i fotogrammi: il veicolo è tra i primi cinque? |
| 5. VLM | Una targa troppo piccola, poi a piena risoluzione |

**Esecuzione:** installate le librerie e scaricate i modelli come dice il `README.md` di questa cartella, poi eseguite
le celle dall'alto con **Maiusc+Invio**. Le celle **Tocca a voi** contengono un esempio già eseguibile: cambiatelo e
rieseguite la cella. La parte 2 usa i risultati della 1; le parti 3, 4 e 5 partono da sole: chi resta indietro salta
alla parte successiva.

## 1. Rilevamento

**La domanda:** quanti veicoli veri trova il modello, e quanti dei suoi riquadri sono giusti?

Il test set: 29 fotogrammi da COCO, una raccolta pubblica di foto usata per misurare i modelli di visione. Per ogni
foto delle persone hanno disegnato i riquadri di auto, camion e autobus: è la verità con cui si confronta il modello.

In [ ]:
# Importate da visione.py, qui accanto, i dati, il modello di rilevamento e le funzioni per misurare e disegnare.
from visione import (
    SOGLIE,
    ModelloDiRilevamento,
    anteprima,
    disegna_esiti,
    disegna_riquadri,
    fotogrammi,
    misura_rilevamento,
    veicoli_veri,
)

# I fotogrammi con il loro gruppo (giorno o notte) e, per ognuno, i riquadri veri disegnati da persone.
foto = fotogrammi()
veri = veicoli_veri()
print(len(foto), "fotogrammi,", sum(len(riquadri) for riquadri in veri.values()), "veicoli veri\n")

# Un campione: un fotogramma di giorno e uno di notte, con i riquadri veri in blu.
for file in ("F07.jpg", "F28.jpg"):
    print(f"{file}   {foto[file]}   {len(veri[file])} veicoli veri, il primo: {veri[file][0]}")
    display(anteprima(disegna_riquadri(file, veri[file]), 480))

Il modello di rilevamento è RT-DETR, addestrato su altre foto della stessa raccolta: queste non le ha mai viste. Per
ogni veicolo restituisce un riquadro e un punteggio da 0 a 1. La prima volta la cella carica il modello (qualche
secondo).

In [ ]:
# Caricate il modello ed eseguitelo una volta su ogni fotogramma, tenendo i riquadri con punteggio da 0,3 in su:
# tutte le misure qui sotto riusano questi rilevamenti.
modello = ModelloDiRilevamento()
rilevamenti = {file: modello(file) for file in foto}

# Cinque dei veicoli trovati in un fotogramma.
file_esempio = "F07.jpg"
for riquadro, punteggio in rilevamenti[file_esempio][:5]:
    print(f"punteggio {punteggio:.2f}   riquadro {riquadro}")

Ogni riga è un veicolo trovato in `F07.jpg`: il punteggio, e il riquadro in pixel (sinistra, alto, destra, basso).

Un riquadro trovato sopra la soglia è giusto (TP) se si sovrappone a un veicolo vero per almeno metà: IoU
(sovrapposizione) di almeno 0,5. Altrimenti è un falso allarme (FP): un riquadro su niente, un doppione dello stesso
veicolo, o uno che lo copre per meno di metà. Un veicolo vero che nessun riquadro copre è un FN.

In [ ]:
# Lo stesso fotogramma con i riquadri veri e quelli trovati sopra la soglia di 0,5, colorati secondo l'esito.
disegna_esiti(file_esempio, rilevamenti[file_esempio], veri[file_esempio], soglia=0.5)

Verde: TP. Rosso: FP. Giallo: FN, il veicolo che il modello non ha trovato.

**Tocca a voi.** Guardate un altro fotogramma, da `F01.jpg` a `F29.jpg`, e cambiate la soglia. Che cosa cambia?

In [ ]:
# Un fotogramma e una soglia a scelta.
file_scelto = "F15.jpg"  # Provate un altro fotogramma, da F01.jpg a F29.jpg.
soglia_scelta = 0.3  # Provate 0.5 e 0.7.
disegna_esiti(file_scelto, rilevamenti[file_scelto], veri[file_scelto], soglia=soglia_scelta)

<details>
<summary>Risposta</summary>

Abbassando la soglia compaiono più riquadri: qualche veicolo giallo diventa verde, ma arrivano molti più rossi.
Alzandola spariscono rossi, e anche qualche verde, che torna giallo.

</details>

Adesso tutto il test set, a tre soglie. Due funzioni di supporto stampano la tabella: `stampa_intestazione` i nomi
delle colonne, `stampa_misure` una riga di conteggi e misure.

In [ ]:
# Definite le due funzioni che stampano la tabella delle misure.
def stampa_intestazione(prima_colonna):
    """I nomi delle colonne; la prima dice che cosa distingue le righe, la soglia o il gruppo."""
    print(
        f"{prima_colonna:<8} {'fotogrammi':>10} {'veicoli':>7} {'TP':>4} {'FP':>4} {'FN':>4}"
        f" {'precisione':>10} {'recall':>6}"
    )


def stampa_misure(etichetta, misure):
    """Una riga della tabella, dal dizionario di `misura_rilevamento`: i conteggi, poi precisione e recall."""
    print(
        f"{etichetta:<8} {misure['fotogrammi']:>10} {misure['veicoli']:>7} {misure['TP']:>4} {misure['FP']:>4}"
        f" {misure['FN']:>4} {misure['precisione']:>10.2f} {misure['recall']:>6.2f}"
    )


# Contate TP, FP e FN su tutti i fotogrammi, una volta per soglia.
stampa_intestazione("soglia")
per_soglia = {soglia: misura_rilevamento(rilevamenti, veri, soglia) for soglia in SOGLIE}
for soglia, misure in per_soglia.items():
    stampa_misure(str(soglia), misure)

- Alzando la soglia, la precisione sale o scende? E la recall?
- Per il triage di foto con armi quale soglia scegliereste? E per un allarme che fa fermare un'auto?
- Con la soglia a 0,3 i FP sono tanti: aprite due fotogrammi con `disegna_esiti(...)`. Sono oggetti scambiati
  per veicoli, doppioni, o riquadri messi male?

<details>
<summary>Risposta</summary>

- **Precisione e recall.** Alzando la soglia la precisione sale (0,45, 0,82, 0,94) e la recall scende (0,84, 0,76,
  0,63).
- **Quale soglia.** Per il triage di foto con armi, una soglia bassa: un'arma mancata costa più di una foto in più da
  guardare. Per un allarme che fa fermare un'auto, una soglia alta: ogni falso allarme ferma una persona.
- **I FP a 0,3.** Soprattutto doppioni, un secondo riquadro sullo stesso veicolo (il pick-up di F15), e auto piccole
  e lontane, dove il riquadro trovato copre quello vero per meno di metà (F18, F24). Pochi sono veicoli che nessuno ha
  disegnato: COCO riquadra quasi tutti i veicoli di ogni foto.

</details>

## 2. Gruppi

**La domanda:** il modello trova i veicoli di notte come di giorno?

La stessa misura divisa per gruppo, alla soglia di 0,5: 24 fotogrammi di giorno, 5 di notte o al crepuscolo. Accanto
alla recall, quanti veicoli ha ogni gruppo.

In [ ]:
# Le stesse misure, una volta per gruppo, alla soglia di 0,5.
SOGLIA = 0.5
per_gruppo = {}
stampa_intestazione("gruppo")
for gruppo in ("giorno", "notte"):
    fotogrammi_del_gruppo = [file for file, gruppo_del_file in foto.items() if gruppo_del_file == gruppo]
    per_gruppo[gruppo] = misura_rilevamento(rilevamenti, veri, SOGLIA, fotogrammi_del_gruppo)
    stampa_misure(gruppo, per_gruppo[gruppo])

# Quanto pesa un singolo veicolo nella recall di notte.
veicoli_di_notte = per_gruppo["notte"]["veicoli"]
print(f"\nDi notte un veicolo trovato in più o in meno sposta la recall di {1 / veicoli_di_notte:.2f}.")

- La recall di notte è più bassa? Di quanto?
- Con quanti veicoli è misurata? Basta per dire che il modello vede peggio di notte?
- Guardate i FN di notte, in giallo qui sotto: sono veicoli al buio, o piccoli e lontani? I fotogrammi di notte
  cambiano anche in altro: pioggia, distanza, tipo di veicolo.

In [ ]:
# I fotogrammi di notte con gli esiti alla soglia di 0,5, rimpiccioliti a 480 pixel per stare nella pagina.
for file, gruppo in foto.items():
    if gruppo == "notte":
        display(anteprima(disegna_esiti(file, rilevamenti[file], veri[file], SOGLIA), 480))

<details>
<summary>Risposta</summary>

- **Di quanto.** La recall di notte è 0,52, quella di giorno 0,80.
- **Su quanti veicoli.** 25, in 5 fotogrammi: un veicolo in più o in meno sposta la recall di 0,04. Non basta per
  dire che il modello vede peggio di notte.
- **I FN di notte.** Dei 12, 11 vengono da due fotogrammi, F28 (5) e F29 (6), e sono auto piccole e lontane: al buio
  e lontane insieme. Con cinque fotogrammi non si separa l'effetto della luce da quello della distanza: è un indizio,
  non una conclusione. Per concludere servirebbero più fotogrammi di notte, con veicoli a tutte le distanze.

</details>

## 3. OCR

**La domanda:** quanti caratteri sbaglia l'OCR, e un IBAN letto è esatto?

Tre documenti del fascicolo stampati come scansioni, un po' storti, sfocati e con un timbro: DOC-01, DOC-05
(l'estratto conto) e DOC-10. Poi FOTO-01: il primo IBAN di DOC-05 scritto a mano e fotografato.

In [ ]:
# La parte 3 parte da sola: importate tutto ciò che le serve.
from visione import (
    DATI,
    FASCICOLO,
    FOTO_IBAN,
    LettoreOCR,
    anteprima,
    apri_immagine,
    caratteri_da_correggere,
    caratteri_incerti,
    cer,
    iban_piu_simile,
    iban_vero,
    riga_che_inizia_con,
    testo_letto,
)

# Un campione: la scansione di DOC-01 e l'inizio del suo testo vero, salvato accanto all'immagine: è il riferimento
# del CER.
display(anteprima(apri_immagine("DOC-01.jpg"), 480))
testo_vero = (DATI / "scansioni" / "DOC-01.txt").read_text(encoding="utf-8")
print("\n".join(testo_vero.splitlines()[:4]))

L'OCR lavora in due passi: trova le zone di testo, poi le legge carattere per carattere. A ogni carattere dà una
confidenza.

In [ ]:
# Caricate l'OCR e mostrate la parte scritta di DOC-05, l'estratto conto.
lettore = LettoreOCR()
estratto_conto = "DOC-05"
display(anteprima(apri_immagine(f"{estratto_conto}.jpg").crop((0, 0, 1240, 560)), 900))

# Leggete la pagina: il risultato sono righe di parole, con una confidenza per ogni carattere.
letture = {estratto_conto: lettore(DATI / "scansioni" / f"{estratto_conto}.jpg")}
print(testo_letto(letture[estratto_conto]))

Confrontate il testo letto con l'immagine. Il tasso di errore sui caratteri (CER) conta i caratteri sostituiti,
cancellati e inseriti, diviso i caratteri del testo vero. I caratteri incerti sono quelli con confidenza sotto 0,9.

In [ ]:
# Leggete le tre scansioni e confrontate ogni testo letto con quello vero, salvato accanto all'immagine.
cer_per_documento = {}
print("documento   CER    caratteri incerti")
for documento in ("DOC-01", "DOC-05", "DOC-10"):
    letture[documento] = lettore(DATI / "scansioni" / f"{documento}.jpg")
    testo_vero = (DATI / "scansioni" / f"{documento}.txt").read_text(encoding="utf-8")
    cer_per_documento[documento] = cer(testo_letto(letture[documento]), testo_vero)
    print(f"{documento}      {cer_per_documento[documento]:.1%}   {len(caratteri_incerti(letture[documento]))}")

Un CER del 3% sembra poco. Ma un IBAN conta solo se ogni carattere è giusto: la corrispondenza esatta. DOC-05 ha due
IBAN: quelli dei bonifici del 3 e del 5 marzo.

In [ ]:
# Il CSV dell'estratto conto, da cui vengono gli IBAN veri: l'intestazione e le righe 6 e 8, i due bonifici.
righe_csv = (FASCICOLO / "core" / "DOC-05-estratto-conto.csv").read_text(encoding="utf-8").splitlines()
print("\n".join(righe_csv[numero - 1] for numero in (3, 6, 8)))

In [ ]:
# Definite la funzione che confronta un IBAN letto con quello vero.
def confronta_iban(nome, testo, iban_atteso):
    """Stampa l'IBAN vero e il pezzo di testo letto più simile; True se coincidono carattere per carattere."""
    letto = iban_piu_simile(testo, iban_atteso)
    da_correggere = caratteri_da_correggere(letto, iban_atteso)
    print(f"{nome}\n  vero:  {iban_atteso.replace(' ', '')}\n  letto: {letto}")
    print(f"  caratteri da correggere: {da_correggere}   esatto: {'sì' if da_correggere == 0 else 'no'}\n")
    return da_correggere == 0


# I due bonifici: il nome, la data con cui comincia la riga letta, la riga del CSV dell'estratto conto con l'IBAN vero.
BONIFICI = [("DOC-05, bonifico del 3 marzo", "03/03/2025", 6), ("DOC-05, bonifico del 5 marzo", "05/03/2025", 8)]
iban_esatti = {}
for nome, data, riga_estratto in BONIFICI:
    iban_esatti[nome] = confronta_iban(nome, riga_che_inizia_con(letture["DOC-05"], data), iban_vero(riga_estratto))

# Le ultime righe della tabella letta, con i caratteri sotto la confidenza di 0,9 tra parentesi quadre.
print("Le righe della tabella, con i caratteri incerti tra parentesi quadre:")
print("\n".join(testo_letto(letture["DOC-05"], soglia=0.9).splitlines()[-5:]))

Adesso FOTO-01, scritto a mano: è lo stesso IBAN del bonifico del 3 marzo.

In [ ]:
# FOTO-01: l'IBAN scritto a mano e fotografato.
from PIL import Image

display(anteprima(Image.open(FOTO_IBAN), 900))

In [ ]:
# Leggete FOTO-01 con lo stesso OCR.
letture["FOTO-01"] = lettore(FOTO_IBAN)

# L'IBAN vero è quello del bonifico del 3 marzo, riga 6 del CSV dell'estratto conto.
iban_esatti["FOTO-01"] = confronta_iban("FOTO-01", testo_letto(letture["FOTO-01"]), iban_vero(6))
print("Con i caratteri incerti tra parentesi quadre:")
print(testo_letto(letture["FOTO-01"], soglia=0.9))

- Quanti IBAN su tre sono esatti?
- I caratteri sbagliati sono tra quelli incerti? Una persona che controlla solo gli incerti li avrebbe trovati tutti?
- L'IBAN del 5 marzo ha due cifre scambiate fin dall'origine: è l'errore del fascicolo. L'OCR l'ha letto com'è
  scritto, o l'ha «corretto»?

<details>
<summary>Risposta</summary>

- **IBAN esatti.** Due su tre: i due stampati di DOC-05. FOTO-01, scritto a mano, è letto
  `IT87 D389 3339 8330 0000 0123 456`: otto caratteri da correggere su 27.
- **Gli incerti.** Solo in parte: gran parte dei 9 letti come 3 o 8 sono tra parentesi quadre, ma l'8 di `D[3]89` no.
  Chi controlla solo gli incerti non trova tutti gli errori.
- **L'IBAN del 5 marzo.** L'OCR legge le cifre scambiate così come sono scritte: non «corregge», ed è ciò che si
  vuole da un OCR. Un modello linguistico, invece, tende a restituire l'IBAN più probabile.

</details>

## 4. Ricerca per immagini

**La domanda:** una frase trova il fotogramma giusto tra i primi cinque?

Il modello (SigLIP 2) mette testo e immagini nello stesso spazio: ogni fotogramma diventa un vettore, e anche una
frase. I fotogrammi più vicini alla frase vengono per primi. È addestrato su coppie di immagini e didascalie in molte
lingue: la frase si scrive in italiano.

In [ ]:
# La parte 4 parte da sola: importate il modello di ricerca e rileggete l'elenco dei fotogrammi.
from visione import RicercaPerImmagini, affianca_fotogrammi, fotogrammi

foto = fotogrammi()

# Caricate il modello e trasformate ogni fotogramma in un vettore, una volta sola.
ricerca = RicercaPerImmagini()
ricerca.indicizza(list(foto))


def cerca_e_mostra(frase, k=5):
    """Stampa i k fotogrammi più vicini alla frase, ognuno con il suo punteggio, e li mostra in fila."""
    trovati = ricerca.cerca(frase, k)
    print(frase, "->", ", ".join(f"{file} ({punteggio:.2f})" for file, punteggio in trovati))
    display(affianca_fotogrammi([file for file, _ in trovati]))


cerca_e_mostra("un furgone dei gelati")

Il banco: cinque frasi, ciascuna con i suoi fotogrammi giusti. Il veicolo cercato è tra i primi cinque (recall@5)? E al
primo posto?

In [ ]:
# Le cinque frasi del banco, ognuna con i fotogrammi che la ricerca dovrebbe trovare.
BANCO = {
    "un furgone dei gelati": {"F23.jpg"},
    "un autobus rosso a due piani": {"F14.jpg", "F19.jpg"},
    "uno scuolabus giallo": {"F12.jpg"},
    "un taxi giallo": {"F01.jpg"},
    "un tram": {"F22.jpg"},
}

# Per ogni frase: un fotogramma giusto è tra i primi cinque? È il primo?
nei_primi_5 = al_primo_posto = 0
for frase, giusti in BANCO.items():
    primi = [file for file, _ in ricerca.cerca(frase, 5)]
    trovato = bool(giusti & set(primi))
    nei_primi_5 += trovato
    al_primo_posto += primi[0] in giusti
    print(f"{frase:<30} primo: {primi[0]}   nei primi 5: {'sì' if trovato else 'no'}")
print(f"\nrecall@5: {nei_primi_5} su {len(BANCO)}   al primo posto: {al_primo_posto} su {len(BANCO)}")

<details>
<summary>Risposta</summary>

recall@5: 5 su 5, e 4 su 5 al primo posto. L'eccezione è «un taxi giallo»: al primo posto c'è F26, l'autobus giallo
di notte; il taxi, F01, è secondo.

</details>

**Tocca a voi.**

- Cercate qualcosa che nel test set non c'è, per esempio un'ambulanza. Che cosa torna?
- Riscrivete una frase del banco con altre parole, o in inglese. Il fotogramma giusto resta tra i primi cinque?

In [ ]:
# Una frase a scelta: qualcosa che nel test set non c'è, o una frase del banco riscritta.
cerca_e_mostra("un'ambulanza")  # Provate "un autobus a due piani di colore rosso", o "a yellow taxi".

<details>
<summary>Risposta</summary>

Per «un'ambulanza» tornano cinque fotogrammi con punteggi tra 0,06 e 0,08, contro 0,11-0,14 delle frasi del banco che
trovano il loro fotogramma. Per una frase riscritta il risultato dipende dalle parole: guardate se il fotogramma
giusto resta tra i primi cinque e di quanto cambia il suo punteggio.

</details>

**Nota.** La ricerca restituisce sempre cinque risultati, anche quando nessuno va bene: l'unico segnale è un
punteggio più basso.

## 5. VLM

**La domanda:** che cosa risponde un VLM quando la targa è troppo piccola per leggerla?

Le tre targhe sono di fantasia e non valide: le targhe italiane non usano le lettere I, O, Q e U.

In [ ]:
# La parte 5 parte da sola: importate il VLM, le foto delle targhe e le funzioni per guardarle e ritagliarle.
from visione import (
    ModelloVisivoLinguistico,
    anteprima,
    apri_immagine,
    disegna_riquadri,
    ritaglia_intorno,
    targhe_di_fantasia,
)

# Un campione: la prima foto con il riquadro della targa in blu, poi il ritaglio intorno alla targa a piena
# risoluzione.
targhe = targhe_di_fantasia()
prima = targhe[0]
print(f"{prima['file']}   targa vera: {prima['targa']}")
display(anteprima(disegna_riquadri(prima["file"], [prima["riquadro"]]), 480))
display(ritaglia_intorno(prima["file"], prima["riquadro"]))

Un modello visivo-linguistico (VLM) piccolo: SmolVLM, 256 milioni di parametri. Riceve un'immagine e una domanda,
risponde in parole. Prima di entrare nel modello, l'immagine intera è rimpicciolita a 512 pixel sul lato lungo.

La domanda è in inglese perché il modello è addestrato in inglese: in italiano tende a descrivere la foto invece di
rispondere.

In [ ]:
# Caricate il VLM e fategli la domanda.
vlm = ModelloVisivoLinguistico()
DOMANDA = "Read the license plate. Answer with the plate text only."


def stessa_targa(risposta, targa_vera):
    """True se la risposta, senza spazi né punteggiatura e in maiuscolo, è la targa vera."""
    return "".join(c for c in risposta.upper() if c.isalnum()) == targa_vera.replace(" ", "")


# Per ogni targa, la stessa domanda due volte: sulla foto intera e sul ritaglio a piena risoluzione.
targhe_esatte = {"foto intera": [], "ritaglio nitido": []}
for targa in targhe:
    intera, vicina = apri_immagine(targa["file"]), ritaglia_intorno(targa["file"], targa["riquadro"])
    print(f"{targa['file']}\ntarga vera:       {targa['targa']}")
    for versione, immagine in (("foto intera", intera), ("ritaglio nitido", vicina)):
        risposta = vlm.chiedi(immagine, DOMANDA)
        esatta = stessa_targa(risposta, targa["targa"])
        targhe_esatte[versione].append(esatta)
        print(f"{versione + ':':<17} {risposta}   esatta: {'sì' if esatta else 'no'}")
    print()

- Sulla foto intera il modello ha detto che non si legge, o ha risposto comunque?
- Sul ritaglio le letture sono esatte? Il modello si accorge che le targhe non sono valide?

**Tocca a voi.** Chiedete al modello, sulla stessa foto intera, se la targa si legge. Se risponde di no, perché
prima ha dato una risposta?

In [ ]:
# Sulla foto intera di T1, una domanda diversa: la targa si legge?
vlm.chiedi(apri_immagine("T1.jpg"), "Can you read the license plate? Answer yes or no.")

<!-- cspell:ignore Bluehead -->
<details>
<summary>Risposta</summary>

- **Sulla foto intera** il modello non dice mai che non si legge: risponde `Bluehead.`, `QI361.` e `6039.`. Nessuna
  è una targa completa; `QI361` ha le prime due lettere giuste e il resto sbagliato, una lettura parziale che sembra
  vera.
- **Sul ritaglio** legge esatte tutte e tre le targhe, e nessuna risposta dice che non sono valide: il controllo del
  formato va fatto dopo, in codice.
- **Alla domanda «si legge la targa?»**, sulla stessa foto di T1, risponde `No.`

</details>

**Nota.** Il modello non sa di non vedere: a «leggi la targa» risponde comunque con un testo, che va controllato come
una lettura qualunque.

## Fonti e note sui dati

- Fotogrammi e riquadri veri: [COCO 2017](https://cocodataset.org), set di validazione. Foto con licenza CC BY 2.0 o
  CC BY-SA 2.0, annotazioni CC BY 4.0. Autore, pagina e licenza di ogni foto sono in `dati/crediti.csv`.
- Rilevamento: [RT-DETR v2](https://huggingface.co/PekingU/rtdetr_v2_r18vd), licenza Apache 2.0.
- OCR: [docTR](https://github.com/mindee/doctr), con i modelli `fast_base` e `crnn_vgg16_bn`, licenza Apache 2.0.
- Ricerca per immagini: [SigLIP 2](https://huggingface.co/google/siglip2-base-patch16-224), licenza Apache 2.0.
- VLM: [SmolVLM](https://huggingface.co/HuggingFaceTB/SmolVLM-256M-Instruct), licenza Apache 2.0.
- Scansioni e FOTO-01 vengono dal fascicolo del corso; le targhe sono tre foto di COCO con sopra una targa di
  fantasia.